<h1>Table of Contents<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#groupBy" data-toc-modified-id="groupBy-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>groupBy</a></span><ul class="toc-item"><li><span><a href="#rollup" data-toc-modified-id="rollup-1.1"><span class="toc-item-num">1.1&nbsp;&nbsp;</span>rollup</a></span></li><li><span><a href="#cube" data-toc-modified-id="cube-1.2"><span class="toc-item-num">1.2&nbsp;&nbsp;</span>cube</a></span></li></ul></li><li><span><a href="#pivot" data-toc-modified-id="pivot-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>pivot</a></span></li></ul></div>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType
import os, warnings

warnings.filterwarnings(action="ignore")

In [2]:
spark = (SparkSession.builder
         .appName("03-API.DataFrames-agregations")
         .getOrCreate())

print("Master :", spark.sparkContext.master)
print("Application :", spark.sparkContext.applicationId)
print("Python :", os.sys.version.split()[0])
print("Spark :", spark.version)

26/09/24 10:01:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/24 10:01:38 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/09/24 10:01:39 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to spark-events/eventlog_v2_app-20260924100137-0019/events_1_app-20260924100137-0019.zstd. This is Unsupported


Master : spark://spark-master:7077
Application : app-20260924100137-0019
Python : 3.10.12
Spark : 4.0.4


In [3]:
spark

In [4]:
print(f"spark.executor.cores = {spark.conf.get('spark.executor.cores')}\nspark.executor.memory = {spark.conf.get('spark.executor.memory')}")

spark.executor.cores = 1
spark.executor.memory = 1g


In [5]:
from pyspark.sql.functions import *

meteo = spark.read.format('csv')\
    .option('sep',';')\
    .option('header','true')\
    .option('nullValue','mq')\
    .option('inferSchema', 'true')\
    .load('../data/meteo/') #    .cache()

meteo = meteo.select(
                 col('numer_sta'),
                 col('date')[0:4].cast('int') ,
                 col('date')[5:2].cast('int'),
                 col('date')[7:2].cast('int'),
                 col('date')[5:4],
                 round(col('t') - 273.15,2),
                 col('u') / 100 ,
                 col('vv') / 1000 ,
                 col('pres') / 1000,
                 coalesce( col('rr3'),
                           col('rr24')/8,
                           col('rr12')/4,
                           col('rr6')/2,
                           col('rr1')*3  ) )\
             .toDF('id','annee','mois','jour','mois_jour','temperature',
                   'humidite','visibilite','pression','precipitations') #             .cache()


from pyspark.sql.types     import StructType, \
     StructField, FloatType, \
     IntegerType, StringType

schema = StructType([
        StructField('Id'           , StringType() , True),
        StructField('ville'        , StringType() , True),
        StructField('latitude'     , FloatType() , True),
        StructField('longitude'    , FloatType() , True),
        StructField('altitude'     , IntegerType() , True)])

villes  = spark.read.format('csv')   \
      .option('sep',';')                \
      .option('mergeSchema', 'true')    \
      .option('header','true')          \
      .schema(schema)                   \
      .load('../data/postesSynop.csv')  \
      .cache()

meteo.count(), villes.count()

(525327, 62)

# groupBy

<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-03.png" width="400">    

In [6]:
meteo.where('id < 8000')\
     .select('annee','mois_jour','temperature','precipitations')\
     .describe().show()

26/09/24 10:01:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 10:=============================>                            (2 + 2) / 4]

+-------+-----------------+-----------------+------------------+--------------------+
|summary|            annee|        mois_jour|       temperature|      precipitations|
+-------+-----------------+-----------------+------------------+--------------------+
|  count|           364194|           364194|            354933|              354170|
|   mean|2023.998388221662|668.2302564018079|13.521904134019707|  0.2547175085411319|
| stddev|0.814986453610176|345.3289387380613| 7.255302188633707|  1.2488533777328372|
|    min|             2023|             0101|             -12.1|-0.30000000000000004|
|    max|             2025|             1231|              42.3|                63.0|
+-------+-----------------+-----------------+------------------+--------------------+



In [7]:
meteo.where('id < 8000')\
     .select('annee','mois_jour','temperature')\
     .describe().show()

[Stage 13:=============================>                            (2 + 2) / 4]

+-------+-----------------+-----------------+------------------+
|summary|            annee|        mois_jour|       temperature|
+-------+-----------------+-----------------+------------------+
|  count|           364194|           364194|            354933|
|   mean|2023.998388221662|668.2302564018079|13.521904134019707|
| stddev|0.814986453610176|345.3289387380613| 7.255302188633707|
|    min|             2023|             0101|             -12.1|
|    max|             2025|             1231|              42.3|
+-------+-----------------+-----------------+------------------+



In [8]:
meteo.where('id < 8000').count()

364194

In [9]:
meteo.where('id < 8000')\
     .select('humidite','visibilite','pression')\
     .describe().show()

[Stage 19:=============================>                            (2 + 2) / 4]

+-------+-------------------+------------------+-----------------+
|summary|           humidite|        visibilite|         pression|
+-------+-------------------+------------------+-----------------+
|  count|             351765|            333567|           352120|
|   mean| 0.7621383878441705| 25.83962820063111|99.54189679654704|
| stddev|0.17497304546478953|16.901574572459065|2.548281213637353|
|    min|               0.01|               0.0|             88.7|
|    max|                1.0|              80.0|           104.42|
+-------+-------------------+------------------+-----------------+



In [10]:
meteo.where('id < 8000')\
     .groupBy('annee')\
     .avg('temperature','visibilite','pression').show(5)

[Stage 22:=============================>                            (2 + 2) / 4]

+-----+------------------+------------------+-----------------+
|annee|  avg(temperature)|   avg(visibilite)|    avg(pression)|
+-----+------------------+------------------+-----------------+
| 2024|13.311918358448018|24.839640450644637| 99.5395800206962|
| 2025|13.476741393114494|27.001192538977445|  99.534842285704|
| 2023|13.778459340176196|25.703045625211246|99.55107509253425|
+-----+------------------+------------------+-----------------+



In [11]:
meteo.where('id < 8000')\
     .groupBy('id','annee')\
     .max('temperature','visibilite','pression').show(5)

[Stage 25:=============================>                            (2 + 2) / 4]

+----+-----+----------------+---------------+-------------+
|  id|annee|max(temperature)|max(visibilite)|max(pression)|
+----+-----+----------------+---------------+-------------+
|7005| 2024|            33.4|           20.0|       102.77|
|7015| 2024|            34.5|           60.0|       103.08|
|7027| 2024|            32.1|           60.0|       102.88|
|7117| 2024|            26.2|           50.0|       102.99|
|7149| 2024|            36.3|           60.0|       102.72|
+----+-----+----------------+---------------+-------------+
only showing top 5 rows


In [12]:
meteo.where('id < 8000')\
     .groupBy('id','annee')\
     .agg(
            count('id').alias('nb_villes'),
            round(avg('temperature'),2).alias('temperature'),
            round(avg('humidite'),2).alias('humidite'),
            round(avg('visibilite'),2).alias('visibilite'),
            round(avg('pression'),2).alias('pression'),
            round(sum('pression')).alias('precipitations'))\
     .orderBy("id","annee")\
     .show(28)

[Stage 28:=============================>                            (2 + 2) / 4]

+----+-----+---------+-----------+--------+----------+--------+--------------+
|  id|annee|nb_villes|temperature|humidite|visibilite|pression|precipitations|
+----+-----+---------+-----------+--------+----------+--------+--------------+
|7005| 2023|     2903|      12.08|    0.82|     16.43|  100.67|      292131.0|
|7005| 2024|     2926|      11.74|    0.83|     16.05|  100.69|      294607.0|
|7005| 2025|     2794|      11.39|    0.81|      16.4|  100.81|      281670.0|
|7015| 2023|     2864|      12.43|    0.78|      21.8|  100.94|      289093.0|
|7015| 2024|     2927|      12.15|    0.81|     21.87|  100.96|      295518.0|
|7015| 2025|     2889|      12.13|    0.76|     20.95|  101.11|      292097.0|
|7020| 2023|     2848|      13.04|    0.83|     12.51|  101.43|      288868.0|
|7020| 2024|     2924|      12.89|    0.83|     12.88|  101.41|      296511.0|
|7020| 2025|     2902|      12.96|    0.83|     12.11|   101.5|      294543.0|
|7027| 2023|     2896|      12.45|    0.81|      24.

In [13]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .orderBy('annee','mois')\
     .show(5)

[Stage 31:=============================>                            (2 + 2) / 4]

+-----+----+--------------+
|annee|mois|precipitations|
+-----+----+--------------+
| 2023|   1|        1008.0|
| 2023|   2|         900.0|
| 2023|   3|         991.0|
| 2023|   4|         975.0|
| 2023|   5|        1008.0|
+-----+----+--------------+
only showing top 5 rows


## rollup
<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-05.png" width="400">

In [14]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .rollup('annee','mois')\
     .agg( round(sum('precipitations')).alias('precipitations'))\
     .orderBy(col('annee').asc_nulls_last(),
              col('mois').asc_nulls_last())\
     .show(14)

[Stage 34:=============================>                            (2 + 2) / 4]

+-----+----+--------------+
|annee|mois|precipitations|
+-----+----+--------------+
| 2023|   1|        1008.0|
| 2023|   2|         900.0|
| 2023|   3|         991.0|
| 2023|   4|         975.0|
| 2023|   5|        1008.0|
| 2023|   6|         970.0|
| 2023|   7|         975.0|
| 2023|   8|        1004.0|
| 2023|   9|         975.0|
| 2023|  10|         996.0|
| 2023|  11|         968.0|
| 2023|  12|        1011.0|
| 2023|NULL|       11781.0|
| 2024|   1|        1013.0|
+-----+----+--------------+
only showing top 14 rows


In [15]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .orderBy('annee','mois')\
     .show()

+-----+----+--------------+
|annee|mois|precipitations|
+-----+----+--------------+
| 2023|   1|        1008.0|
| 2023|   2|         900.0|
| 2023|   3|         991.0|
| 2023|   4|         975.0|
| 2023|   5|        1008.0|
| 2023|   6|         970.0|
| 2023|   7|         975.0|
| 2023|   8|        1004.0|
| 2023|   9|         975.0|
| 2023|  10|         996.0|
| 2023|  11|         968.0|
| 2023|  12|        1011.0|
| 2024|   1|        1013.0|
| 2024|   2|         938.0|
| 2024|   3|        1001.0|
| 2024|   4|         972.0|
| 2024|   5|        1003.0|
| 2024|   6|         967.0|
| 2024|   7|        1006.0|
| 2024|   8|        1011.0|
+-----+----+--------------+
only showing top 20 rows


In [16]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .rollup('annee','mois')\
     .agg( round(sum('precipitations')).alias('precipitations'))\
     .orderBy('annee','mois')\
     .toPandas().head(10) #.show()

,annee,mois,precipitations
0,NaN,NaN,35050.0
1,2023.0,NaN,11781.0
2,2023.0,1.0,1008.0
3,2023.0,2.0,900.0
4,2023.0,3.0,991.0
5,2023.0,4.0,975.0
6,2023.0,5.0,1008.0
7,2023.0,6.0,970.0
8,2023.0,7.0,975.0
9,2023.0,8.0,1004.0


In [17]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .rollup('annee','mois')\
     .agg(round(sum('precipitations')).alias('precipitations'))\
     .orderBy('annee','mois')\
     .toPandas().head(20) #.show(20)

,annee,mois,precipitations
0,NaN,NaN,35050.0
1,2023.0,NaN,11781.0
2,2023.0,1.0,1008.0
3,2023.0,2.0,900.0
4,2023.0,3.0,991.0
5,2023.0,4.0,975.0
6,2023.0,5.0,1008.0
7,2023.0,6.0,970.0
8,2023.0,7.0,975.0
9,2023.0,8.0,1004.0


In [18]:
meteo.where('id < 8000')\
     .rollup('annee','mois')\
     .agg(
            round(sum('precipitations')).alias('precipitations'))\
     .orderBy(col('annee').asc_nulls_last(),
              col('mois').asc_nulls_last())\
     .toPandas().head(16) # .show(16)

,annee,mois,precipitations
0,2023.0,1.0,2391.0
1,2023.0,2.0,470.0
2,2023.0,3.0,2761.0
3,2023.0,4.0,2043.0
4,2023.0,5.0,2131.0
5,2023.0,6.0,2490.0
6,2023.0,7.0,1729.0
7,2023.0,8.0,1981.0
8,2023.0,9.0,1838.0
9,2023.0,10.0,3881.0


In [19]:
meteo.where('id < 8000')\
     .rollup('annee','mois')\
     .agg(
            round(avg('temperature'),2).alias('temperature'),
            round(avg('humidite'),2).alias('humidite'),
            round(avg('visibilite'),2).alias('visibilite'),
            round(avg('pression'),2).alias('pression'),
            round(avg('precipitations'),2).alias('precipitations'))\
     .orderBy(col('annee').asc_nulls_last(),
              col('mois').asc_nulls_last())\
     .toPandas().head(16) #.show(16)

,annee,mois,temperature,humidite,visibilite,pression,precipitations
0,2023.0,1.0,6.08,0.82,24.25,99.81,0.24
1,2023.0,2.0,6.32,0.76,22.66,100.58,0.05
2,2023.0,3.0,9.62,0.74,26.11,99.23,0.28
3,2023.0,4.0,11.21,0.73,27.18,99.49,0.21
4,2023.0,5.0,15.38,0.73,24.30,99.75,0.21
5,2023.0,6.0,20.64,0.69,23.82,99.50,0.26
6,2023.0,7.0,21.13,0.68,26.88,99.48,0.18
7,2023.0,8.0,21.04,0.70,28.40,99.49,0.20
8,2023.0,9.0,20.15,0.71,28.40,99.56,0.19
9,2023.0,10.0,15.73,0.76,27.38,99.08,0.39


## cube
<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-06.png" width="400">

In [20]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('precipitations'))\
     .cube('annee','mois')\
     .agg( round(sum('precipitations')).alias('precipitations'))\
     .orderBy(col('annee'),col('mois'))\
     .toPandas().head(24) #.show(24)

,annee,mois,precipitations
0,NaN,NaN,35050.0
1,NaN,1.0,2999.0
2,NaN,2.0,2718.0
3,NaN,3.0,2957.0
4,NaN,4.0,2887.0
5,NaN,5.0,2986.0
6,NaN,6.0,2876.0
7,NaN,7.0,2953.0
8,NaN,8.0,2975.0
9,NaN,9.0,2895.0


In [21]:
meteo.where('id < 8000')\
     .groupBy('id','annee')\
     .agg(
            {'id':'count',
            'temperature':'avg',
            'humidite':'avg'}
     ).toPandas().head(10) #.show(10)

,id,annee,avg(humidite),avg(temperature),count(id)
0,7005,2024,0.831439,11.741866,2926
1,7015,2024,0.813157,12.151589,2927
2,7027,2024,0.826264,12.068560,2923
3,7117,2024,0.825930,12.823256,2924
4,7149,2024,0.782771,12.738094,2927
5,7190,2024,0.793995,12.521975,2926
6,7460,2024,0.742785,12.933527,2926
7,7510,2024,0.793691,14.457587,2926
8,7591,2024,0.627204,11.697256,2915
9,7650,2024,0.680875,16.554614,2926


In [22]:
meteo.where('id < 8000')\
     .groupBy('id','annee')\
     .agg(
            {'id':'count',
            'temperature':'avg',
            'humidite':'avg'}
     ).toDF('id','annee','humidite','temperature','nb_villes').toPandas().head(10) #.show(10)

,id,annee,humidite,temperature,nb_villes
0,7015,2023,0.783642,12.430796,2864
1,7149,2023,0.728248,13.498107,2906
2,7190,2023,0.731619,13.006144,2906
3,7335,2023,0.768168,13.627398,2900
4,7471,2023,0.727744,10.518429,2903
5,7481,2023,0.683284,14.397900,2905
6,7558,2023,0.735973,12.073557,2894
7,7621,2023,0.771039,13.701308,2906
8,7005,2024,0.831439,11.741866,2926
9,7015,2024,0.813157,12.151589,2927


In [23]:
meteo.where('id < 8000')\
     .groupBy('id')\
     .agg(
        round(skewness  ('temperature'),3).alias('skewness'  ),
        round(kurtosis  ('temperature'),3).alias('kurtosis'  ),
        round(variance  ('temperature'),3).alias('variance'  ),
        round(var_pop   ('temperature'),3).alias('var_pop'   ),
        round(stddev    ('temperature'),3).alias('stddev'    ),
        round(stddev_pop('temperature'),3).alias('stddev_pop'))\
     .orderBy('id')\
     .toPandas().head(15) #.show(15)

,id,skewness,kurtosis,variance,var_pop,stddev,stddev_pop
0,7005,0.109,-0.014,38.011,38.007,6.165,6.165
1,7015,0.177,-0.204,46.167,46.162,6.795,6.794
2,7020,-0.171,-0.432,16.326,16.324,4.041,4.040
3,7027,0.109,-0.016,35.295,35.291,5.941,5.941
4,7037,0.167,-0.128,41.443,41.438,6.438,6.437
5,7072,0.189,-0.179,54.282,54.276,7.368,7.367
6,7110,0.061,0.251,24.243,24.240,4.924,4.923
7,7117,-0.051,-0.185,18.556,18.554,4.308,4.307
8,7130,0.186,0.102,41.272,41.267,6.424,6.424
9,7139,0.206,-0.064,45.502,45.497,6.746,6.745


In [24]:
meteo.where('id < 8000 and annee > 2014')\
     .groupBy('id','annee')\
     .agg( round(avg('temperature'),2).alias('temperature'))\
     .orderBy("id","annee")\
     .toPandas().head(10) #.show(10)

,id,annee,temperature
0,7005,2023,12.08
1,7005,2024,11.74
2,7005,2025,11.39
3,7015,2023,12.43
4,7015,2024,12.15
5,7015,2025,12.13
6,7020,2023,13.04
7,7020,2024,12.89
8,7020,2025,12.96
9,7027,2023,12.45


# pivot
<img src="https://raw.githubusercontent.com/rbizoi/AnalyserLesDonneesAvecSpark/main/DataFrameSpark/images/M06-04.png" width="400"> 

In [25]:
meteo.where('id < 8000 and annee > 2014')\
      .groupBy('id')\
      .pivot('annee')\
      .agg( round(avg('temperature'),2))\
      .sort('id')\
      .toPandas().head(10) #.show(10)

,id,2023,2024,2025
0,7005,12.08,11.74,11.39
1,7015,12.43,12.15,12.13
2,7020,13.04,12.89,12.96
3,7027,12.45,12.07,12.07
4,7037,11.94,11.37,11.47
5,7072,12.22,11.96,11.81
6,7110,12.57,12.06,12.28
7,7117,13.03,12.82,12.91
8,7130,13.34,12.54,12.99
9,7139,12.36,11.77,12.07


In [26]:
villes.select('ville',
               round('altitude',-2).alias('altitude'))\
      .groupBy('altitude')\
      .agg(collect_list('ville').alias('ville par altitude')).toPandas().head()
# .show(truncate=False)

,altitude,ville par altitude
0,300,"[NANCY-OCHEY, BALE-MULHOUSE, CLERMONT-FD, GOUR..."
1,400,"[LIMOGES-BELLEGARDE, TARBES-OSSUN, ST GIRONS]"
2,900,[EMBRUN]
3,100,"[ABBEVILLE, CAEN-CARPIQUET, REIMS-PRUNAY, BRES..."
4,0,"[LILLE-LESQUIN, PTE DE LA HAGUE, RENNES-ST JAC..."


In [27]:
meteo.where('id < 8000')\
     .groupBy('annee','mois')\
     .agg(round(sum('pression') / 1000).alias('pression'))\
     .orderBy('annee','mois')\
     .toPandas().head()

,annee,mois,pression
0,2023,1,1008.0
1,2023,2,900.0
2,2023,3,991.0
3,2023,4,975.0
4,2023,5,1008.0
